# Orbit Wars — Agent v10 Benchmark Notebook
Run all cells top-to-bottom. Uses `importlib.reload` to always pick up the latest `main.py`.

In [ ]:
import math, importlib, sys
from statistics import mean
from typing import List

try:
    from kaggle_environments import make
    print('kaggle_environments loaded ✓')
except ImportError:
    make = None
    print('kaggle_environments not found — pip install kaggle-environments>=1.28.0')


In [ ]:
# Always reload main.py to avoid Jupyter's namespace cache
if 'main' in sys.modules: importlib.reload(sys.modules['main'])
else: import main
import main
production_agent = main.agent
print('Production agent (v10) loaded ✓')


In [ ]:
# ── Baseline agent (v7) — inline, no file dependency ──────────────────────
import math as _m

_MIN_G = 4
_SFT   = 4
_MLS   = 8

def _fs(n):
    if n<=1: return 1.0
    n=min(max(int(n),1),1000)
    return 1.0+5.0*(_m.log(n)/_m.log(1000))**1.5

def _sa(sx,sy,tx,ty):
    def _hits(x1,y1,x2,y2):
        r=10.0+1.2; dx,dy=x2-x1,y2-y1
        if dx==0 and dy==0: return _m.hypot(x1-50,y1-50)<=r
        t=max(0.,min(1.,((50-x1)*dx+(50-y1)*dy)/(dx*dx+dy*dy)))
        return _m.hypot(x1+t*dx-50,y1+t*dy-50)<=r
    d=_m.atan2(ty-sy,tx-sx)
    if not _hits(sx,sy,tx,ty): return d
    for off in [0.22,-0.22,0.45,-0.45,0.6,-0.6]:
        a=d+off
        if not _hits(sx,sy,sx+200*_m.cos(a),sy+200*_m.sin(a)): return a
    return None

def _pp(sx,sy,tgt,av,n,it=6):
    tx,ty=tgt[2],tgt[3]
    if _m.hypot(tx-50,ty-50)+tgt[4]>=50: return tx,ty
    dx,dy=tx-50,ty-50; r=_m.hypot(dx,dy)
    if r<1e-6: return tx,ty
    th0=_m.atan2(dy,dx); t=0.; sp=max(_fs(n),1e-6)
    for _ in range(it):
        th=th0+av*t; fx=50+r*_m.cos(th); fy=50+r*_m.sin(th)
        t=_m.hypot(sx-fx,sy-fy)/sp
    th=th0+av*t; return 50+r*_m.cos(th),50+r*_m.sin(th)

def baseline_v7(obs, conf=None):
    if isinstance(obs,dict):
        pid=obs.get('player',0); pls=obs.get('planets',[])
        fls=obs.get('fleets',[]); av=obs.get('angular_velocity',0.035)
        cids=set(obs.get('comet_planet_ids',[]or[]))
    else:
        pid=obs.player; pls=obs.planets; fls=getattr(obs,'fleets',[])
        av=getattr(obs,'angular_velocity',0.035)
        cids=set(getattr(obs,'comet_planet_ids',None)or[])
    moves=[]
    mine=[p for p in pls if p[1]==pid]
    foes=[p for p in pls if p[1]!=pid]
    if not mine or not foes: return moves
    tot_prod=sum(p[6] for p in mine)
    def _tsc(sx,sy,t):
        d=_m.log(_m.hypot(sx-t[2],sy-t[3])+2)
        if t[1]==-1: m2=4.0 if tot_prod<15 else 2.0; return (t[6]*m2)/d-0.01*t[5]
        m2=0.5 if tot_prod<12 else 1.5; return (t[6]*m2)/d-0.02*t[5]
    for src in mine:
        sx,sy=src[2],src[3]
        th_in=sum(f[6] for f in fls if f[1]!=pid and _m.hypot(f[2]-sx,f[3]-sy)<80 and
                  abs((f[4]-_m.atan2(sy-f[3],sx-f[2])+_m.pi)%(2*_m.pi)-_m.pi)<0.5)
        res=max(_MIN_G,int(_MIN_G+th_in*1.15))
        avail=src[5]-res
        if avail<_MLS: continue
        ranked=sorted(foes,key=lambda t:_tsc(sx,sy,t),reverse=True)
        for tgt in ranked:
            if avail<_MLS: break
            px,py=_pp(sx,sy,tgt,av,avail)
            tt=_m.hypot(sx-px,sy-py)/max(_fs(avail),1e-6)
            gr=tgt[6]*tt if tgt[1]>=0 else 0
            need=int(tgt[5]+gr+_SFT)
            if avail<need: continue
            launch=min(need,src[5]-_MIN_G)
            if launch<_MLS: continue
            px2,py2=_pp(sx,sy,tgt,av,launch)
            ang=_sa(sx,sy,px2,py2)
            if ang is None: continue
            moves.append([src[0],ang,launch])
            avail-=launch
            break
    return moves

print('Baseline v7 loaded ✓')


In [ ]:
def run_benchmark(num_games=20, base_seed=1348462874):
    if make is None:
        print('kaggle_environments not available.'); return

    print(f'Running {num_games}-game tournament on procedurally varied maps...\n')
    wins=losses=draws=0
    prod_scores=[]; base_scores=[]
    failed_seeds=[]

    for i in range(num_games):
        seed = base_seed + i
        env = make('orbit_wars', configuration={'seed': seed}, debug=False)
        if i % 2 == 0:
            agents = [production_agent, baseline_v7]
            pi, bi = 0, 1
        else:
            agents = [baseline_v7, production_agent]
            pi, bi = 1, 0
        env.run(agents)
        final = env.steps[-1]
        ps = getattr(final[pi],'reward',0) or 0
        bs = getattr(final[bi],'reward',0) or 0
        prod_scores.append(ps); base_scores.append(bs)
        if ps > bs:   wins+=1;   tag='V10 WIN ✓'
        elif bs > ps: losses+=1; tag='BASELINE WIN ✗'; failed_seeds.append(seed)
        else:         draws+=1;  tag='DRAW'
        print(f' Game {i+1:02d} (Seed {seed}): {tag:<22} v10={ps:4}  base={bs:4}')

    wr = wins/num_games*100
    print(f'\n{"─"*62}')
    print(f'Tournament Summary:')
    print(f'Win Rate : {wr:.1f}% ({wins}W / {losses}L / {draws}D)')
    print(f'Avg Score: v10={mean(prod_scores):.2f}  baseline={mean(base_scores):.2f}')
    print(f'{"─"*62}')
    if wr >= 80:  print('🔥 Dominant — submit to Kaggle!')
    elif wr >= 65: print('📈 Strong — submit and iterate.')
    else:          print('⚠️  Needs work.')
    return failed_seeds

failed = run_benchmark(num_games=20)


In [ ]:
# ── Replay export ────────────────────────────────────────────────────────
# Change TARGET_SEED to any seed from `failed` list above
TARGET_SEED = failed[0] if failed else 1348462874

if make is not None:
    env = make('orbit_wars', configuration={'seed': TARGET_SEED}, debug=False)
    env.run([production_agent, baseline_v7])
    html = env.render(mode='html', width=900, height=700)
    fname = f'replay_seed_{TARGET_SEED}.html'
    with open(fname,'w') as f: f.write(html)
    print(f'Replay saved → {fname}')
    env.render(mode='ipython', width=900, height=700)
